# KissanConnect — Crop Disease Classifier: Phase 1 (Model Training)

This notebook trains a crop disease classification model using **transfer learning with MobileNetV2** on the **PlantVillage dataset**.

**Steps:**
1. Setup & GPU check
2. Download dataset from Kaggle
3. Build data pipelines
4. Train (Stage 1: frozen base)
5. Fine-tune (Stage 2: unfrozen top layers)
6. Evaluate & plot results
7. Save model + class names for Phase 2 (Flask API)

**Before running:** Set `Runtime -> Change runtime type -> T4 GPU`.

## 1. Setup & GPU check

In [ ]:
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

if not tf.config.list_physical_devices('GPU'):
    print('\n⚠️  No GPU detected. Go to Runtime -> Change runtime type -> select T4 GPU, then re-run.')

## 2. Download dataset from Kaggle

You need a Kaggle API token:
1. Go to https://www.kaggle.com/settings
2. Click **Create New Token** — this downloads `kaggle.json`
3. Run the cell below and upload that file when prompted

In [ ]:
!pip install -q kaggle

from google.colab import files
print('Upload your kaggle.json file:')
uploaded = files.upload()

In [ ]:
import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

!kaggle datasets download -d abdallahalidev/plantvillage-dataset
!unzip -q plantvillage-dataset.zip -d plantvillage_raw
!echo 'Done. Top-level contents:'
!ls plantvillage_raw

In [ ]:
# Find the actual image directory (dataset has nested folders — this locates the one with class subfolders)
import glob

candidates = glob.glob('plantvillage_raw/**/color', recursive=True)
if not candidates:
    candidates = glob.glob('plantvillage_raw/**', recursive=True)
    candidates = [c for c in candidates if os.path.isdir(c) and len(os.listdir(c)) > 5]

DATA_DIR = candidates[0]
print('Using data directory:', DATA_DIR)
print('Number of classes found:', len(os.listdir(DATA_DIR)))
print('Sample classes:', os.listdir(DATA_DIR)[:5])

## 3. Build data pipelines

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f'Found {num_classes} classes')
print(class_names)

In [ ]:
# Performance: cache + prefetch
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# Data augmentation (helps generalization, applied only during training)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

## 4. Build model — Stage 1: frozen MobileNetV2 base

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras import layers, models

base_model = MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
EPOCHS_STAGE1 = 10

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True, monitor='val_accuracy'),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE1,
    callbacks=callbacks
)

## 5. Fine-tune — Stage 2: unfreeze top layers of the base model

In [ ]:
base_model.trainable = True

# Freeze all layers except the last ~30 — keeps early generic features intact,
# lets later layers adapt to leaf/disease specific patterns
fine_tune_at = len(base_model.layers) - 30
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # much lower LR for fine-tuning
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
EPOCHS_STAGE2 = 8
total_epochs = EPOCHS_STAGE1 + EPOCHS_STAGE2

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=total_epochs,
    initial_epoch=history.epoch[-1] + 1,
    callbacks=callbacks
)

## 6. Evaluate & plot results

In [ ]:
import matplotlib.pyplot as plt

acc = history.history['accuracy'] + history_fine.history['accuracy']
val_acc = history.history['val_accuracy'] + history_fine.history['val_accuracy']
loss = history.history['loss'] + history_fine.history['loss']
val_loss = history.history['val_loss'] + history_fine.history['val_loss']

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(acc, label='Train Accuracy')
plt.plot(val_acc, label='Val Accuracy')
plt.axvline(EPOCHS_STAGE1 - 1, color='gray', linestyle='--', label='Fine-tuning starts')
plt.legend()
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(loss, label='Train Loss')
plt.plot(val_loss, label='Val Loss')
plt.axvline(EPOCHS_STAGE1 - 1, color='gray', linestyle='--', label='Fine-tuning starts')
plt.legend()
plt.title('Loss')

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

final_val_acc = val_acc[-1]
print(f'\nFinal validation accuracy: {final_val_acc:.4f} ({final_val_acc*100:.2f}%)')

In [ ]:
# Confusion matrix / classification report on validation set
import numpy as np
from sklearn.metrics import classification_report

y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

## 7. Save model + class names for Phase 2 (Flask API)

In [ ]:
import json

model.save('kissanconnect_model.h5')

with open('class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)

print('Saved: kissanconnect_model.h5')
print('Saved: class_names.json')
print(f'\nModel trained on {num_classes} classes.')
print(f'Final validation accuracy: {final_val_acc*100:.2f}%')

In [ ]:
# Download the model + class names + training curve to your machine
# (You'll need these for Phase 2 — the Flask backend)
from google.colab import files

files.download('kissanconnect_model.h5')
files.download('class_names.json')
files.download('training_curves.png')

---
### Next: Phase 2
Once this finishes running, come back with:
- The final validation accuracy printed above
- The three downloaded files (`kissanconnect_model.h5`, `class_names.json`, `training_curves.png`)

and we'll build the Flask `/predict` API around this model.